Pip installs for Oracle 

In [2]:
# pip install gymnasium[atari]
# pip install gymnasium[accept-rom-license]
# pip install stable-baselines3

In [1]:
import gymnasium as gym
import ale_py
import numpy as np
from stable_baselines3 import PPO
from gymnasium.wrappers import TransformAction
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import SubprocVecEnv

from utils import Pong6InputWrapper
# Add oracle test train utils

In [ ]:
# import gymnasium as gym
# import ale_py
# from stable_baselines3 import PPO
# from stable_baselines3.common.vec_env import DummyVecEnv
# from gymnasium.wrappers import TransformAction
# import numpy as np

# # 1. Linear Schedule Helper (Keep this for Learning Rate only)
# def linear_schedule(initial_value: float):
#     def func(progress_remaining: float) -> float:
#         return progress_remaining * initial_value
#     return func

# if __name__ == "__main__":
#     # Ensure your custom classes (Pong6InputWrapper, action_mapper) are defined!
#     env = DummyVecEnv([make_env for _ in range(4)])

#     model = PPO(
#         "MlpPolicy",
#         env,
#         # Scheduler is supported here
#         learning_rate=linear_schedule(3e-4),
#         n_steps=2048,
#         batch_size=512,
#         n_epochs=10,
#         gamma=0.99,
#         gae_lambda=0.95,
#         clip_range=0.2,
#         # FIX: ent_coef MUST be a float, not a function
#         ent_coef=0.01, 
#         policy_kwargs=dict(net_arch=[64, 64]),
#         verbose=1,
#         tensorboard_log="./ppo_pong_hyper_oracle/"
#     )

#     print("Starting Precision Training (The Shutout Run)...")
#     # 6M steps to reach peak performance
#     model.learn(total_timesteps=6_000_000, log_interval=25) 
#     model.save("pong_oracle_dnn_final")



In [6]:
class PongEvalWrapper(gym.ObservationWrapper):
    def __init__(self, env):
        super().__init__(env)
        self.observation_space = gym.spaces.Box(low=0, high=1, shape=(6,), dtype=np.float32)
        self.prev_ball_pos = None

    def observation(self, obs):
        # Must match the training scaling exactly: 0.0 to 1.0
        p1_y, p2_y = obs[51]/255.0, obs[50]/255.0
        b_x, b_y   = obs[49]/255.0, obs[54]/255.0

        if self.prev_ball_pos is None:
            v_x, v_y = 0.5, 0.5
        else:
            # Reconstruct the scaled physics seen during training
            v_x = (b_x - self.prev_ball_pos[0]) + 0.5
            v_y = (b_y - self.prev_ball_pos[1]) + 0.5
            # Velocity reset for teleports
            if abs(v_x - 0.5) > 0.1 or abs(v_y - 0.5) > 0.1:
                v_x, v_y = 0.5, 0.5

        self.prev_ball_pos = (b_x, b_y)
        return np.array([p1_y, p2_y, b_x, b_y, v_x, v_y], dtype=np.float32)

    def reset(self, **kwargs):
        self.prev_ball_pos = None
        obs, info = self.env.reset(**kwargs)
        return self.observation(obs), info

# 2. Updated Eval Block with Win Percentage
def evaluate_oracle(model_path, num_episodes=7):
    gym.register_envs(ale_py)
    base_env = gym.make("ALE/Pong-v5", obs_type="ram")
    
    # Use the Eval Wrapper to keep scores 'pure' (No +0.1 paddle-hit bonuses)
    env = PongEvalWrapper(base_env) 
    env = TransformAction(env, 
                          lambda a: [0, 2, 3][a],
                          action_space=gym.spaces.Discrete(3))

    model = PPO.load(model_path)
    all_episode_rewards = []
    wins = 0
    
    print(f"--- Evaluating {model_path} ---")

    for episode in range(num_episodes):
        obs, info = env.reset()
        terminated, truncated = False, False
        episode_reward = 0
        
        while not (terminated or truncated):
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(action)
            episode_reward += reward 
        
        all_episode_rewards.append(episode_reward)
        
        # In Pong, a positive reward means you scored more than the opponent
        if episode_reward > 0:
            wins += 1
            
        print(f"Episode {episode + 1}: True Score = {int(episode_reward)}")

    mean_reward = np.mean(all_episode_rewards)
    win_percentage = (wins / num_episodes) * 100
    
    print("-" * 35)
    print(f"Mean True Score: {mean_reward:.2f}")
    print(f"Win Percentage: {win_percentage:.1f}%")
    print(f"Status: {'Champion' if mean_reward > 15 else 'Challenger'}")
    print("-" * 35)
    
    env.close()
    return mean_reward

# Run it
evaluate_oracle("pong_oracle_dnn_final", num_episodes=15)

--- Evaluating pong_oracle_dnn_final ---
Episode 1: True Score = 14
Episode 2: True Score = 9
Episode 3: True Score = 7
Episode 4: True Score = 9
Episode 5: True Score = 14
Episode 6: True Score = 11
Episode 7: True Score = 15
Episode 8: True Score = 17
Episode 9: True Score = 10
Episode 10: True Score = 12
Episode 11: True Score = 11
Episode 12: True Score = 15
Episode 13: True Score = 4
Episode 14: True Score = 13
Episode 15: True Score = 11
-----------------------------------
Mean True Score: 11.47
Win Percentage: 100.0%
Status: Challenger
-----------------------------------


np.float64(11.466666666666667)

In [ ]:
# Win percent 90%

In [9]:
import pandas as pd
import torch
import gymnasium as gym
from stable_baselines3 import PPO
from gymnasium.wrappers import TransformAction

def generate_expert_dataset(model_path, n_episodes=50, output_file="expert_data.csv"):
    # 1. Setup environment to match training exactly
    base_env = gym.make("ALE/Pong-v5", obs_type="ram")
    env = Pong6InputWrapper(base_env) 
    env = TransformAction(env, lambda a: [0, 2, 3][a], action_space=gym.spaces.Discrete(3))
    
    model = PPO.load(model_path)
    data_rows = []
    
    print(f"Generating data from {n_episodes} games using 3-output Oracle...")
    
    for i in range(n_episodes):
        obs, info = env.reset()
        done = False
        while not done:
            # Get the 3-action distribution [Stay, Up, Down]
            obs_tensor = torch.as_tensor(obs).unsqueeze(0).to(model.device)
            with torch.no_grad():
                dist = model.policy.get_distribution(obs_tensor)
                probs = dist.distribution.probs.cpu().numpy()[0]
            
            # Since the model was trained on the 3-action space:
            # probs[0] = Stay, probs[1] = Up, probs[2] = Down
            p_stay, p_up, p_down = probs[0], probs[1], probs[2]
            
            # Record state + probabilities
            data_rows.append(list(obs) + [p_stay, p_up, p_down])
            
            # Step the game
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            
        if (i+1) % 10 == 0:
            print(f"Processed {i+1}/{n_episodes} games...")

    # Save to CSV
    cols = ['p1_y', 'p2_y', 'b_x', 'b_y', 'v_x', 'v_y', 'target_stay', 'target_up', 'target_down']
    df = pd.DataFrame(data_rows, columns=cols)
    df.to_csv(output_file, index=False)
    print(f"Success: Saved {len(df)} frames to {output_file}")
    env.close()

generate_expert_dataset("pong_oracle_dnn_final", n_episodes=3)

Generating data from 3 games using 3-output Oracle...
Success: Saved 29400 frames to expert_data.csv


In [10]:
from sklearn.tree import DecisionTreeRegressor, export_text

def train_regression_stump(csv_file="expert_data.csv", max_depth=5):
    # Load data
    df = pd.read_csv(csv_file)
    
    X = df[['p1_y', 'p2_y', 'b_x', 'b_y', 'v_x', 'v_y']]
    y = df[['target_stay', 'target_up', 'target_down']]
    
    # Train the regressor
    # A depth of 5 is usually the "knee" of the curve for Pong interpretability
    regr = DecisionTreeRegressor(max_depth=max_depth, random_state=42)
    regr.fit(X, y)
    
    print(f"Stump trained successfully (Depth: {max_depth})")
    
    # Export the logic to a text file for your paper/analysis
    text_repr = export_text(regr, feature_names=list(X.columns))
    print("\n--- EXTRACTED HEURISTICS ---")
    print(text_repr)
    
    return regr

stump_model = train_regression_stump(max_depth=5)

Stump trained successfully (Depth: 5)

--- EXTRACTED HEURISTICS ---
|--- v_y <= 0.50
|   |--- b_x <= 0.80
|   |   |--- v_y <= 0.48
|   |   |   |--- p1_y <= 0.71
|   |   |   |   |--- b_y <= 0.63
|   |   |   |   |   |--- value: [0.60, 0.39, 0.01]
|   |   |   |   |--- b_y >  0.63
|   |   |   |   |   |--- value: [0.83, 0.14, 0.04]
|   |   |   |--- p1_y >  0.71
|   |   |   |   |--- b_x <= 0.59
|   |   |   |   |   |--- value: [0.48, 0.52, 0.01]
|   |   |   |   |--- b_x >  0.59
|   |   |   |   |   |--- value: [0.08, 0.92, 0.00]
|   |   |--- v_y >  0.48
|   |   |   |--- b_x <= 0.21
|   |   |   |   |--- p1_y <= 0.25
|   |   |   |   |   |--- value: [0.78, 0.22, 0.00]
|   |   |   |   |--- p1_y >  0.25
|   |   |   |   |   |--- value: [0.01, 0.99, 0.00]
|   |   |   |--- b_x >  0.21
|   |   |   |   |--- p1_y <= 0.17
|   |   |   |   |   |--- value: [0.19, 0.00, 0.80]
|   |   |   |   |--- p1_y >  0.17
|   |   |   |   |   |--- value: [0.77, 0.15, 0.08]
|   |--- b_x >  0.80
|   |   |--- b_y <= 0.10
|   

In [ ]:
def evaluate_stump(stump_model, num_episodes=7):
    # 1. Setup the exact same environment stack
    base_env = gym.make("ALE/Pong-v5", obs_type="ram")
    # Using the Eval Wrapper (0.0-1.0 scaling, no reward shaping)
    env = PongEvalWrapper(base_env) 
    env = TransformAction(env, 
                          lambda a: [0, 2, 3][a],
                          action_space=gym.spaces.Discrete(3))

    all_episode_rewards = []
    wins = 0
    
    print(f"--- Evaluating Decision Tree Stump ---")

    for episode in range(num_episodes):
        obs, info = env.reset()
        terminated, truncated = False, False
        episode_reward = 0
        
        while not (terminated or truncated):
            # 1. Convert the 1D numpy array to a 2D DataFrame with the expected feature names
            feature_names = ['p1_y', 'p2_y', 'b_x', 'b_y', 'v_x', 'v_y']
            obs_df = pd.DataFrame([obs], columns=feature_names)
            
            # 2. Predict using the DataFrame (silences the warning)
            prob_dist = stump_model.predict(obs_df)
            
            # 3. Argmax to get the best action index
            action = np.argmax(prob_dist[0])
            
            # Step the environment
            obs, reward, terminated, truncated, info = env.step(action)
            episode_reward += reward
        
        all_episode_rewards.append(episode_reward)
        if episode_reward > 0:
            wins += 1
            
        print(f"Episode {episode + 1}: True Score = {int(episode_reward)}")

    mean_reward = np.mean(all_episode_rewards)
    win_percentage = (wins / num_episodes) * 100
    
    print("-" * 35)
    print(f"Mean True Score: {mean_reward:.2f}")
    print(f"Win Percentage: {win_percentage:.1f}%")
    # Calculate performance retention relative to your DNN Oracle's 4.57 Mean Score
    # (Assuming 4.57 was your teacher's baseline)
    print("-" * 35)
    
    env.close()
    return mean_reward

# Usage:
evaluate_stump(stump_model, num_episodes=10)

--- Evaluating Decision Tree Stump ---
Episode 1: True Score = -21
